# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAyyanHassan/flyrank-ml-internship-work/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring.**

This lane fits because staleness and decline are both directly present in the starter dataset (`freshness_tier`, `days_since_last_update`, `trend_direction`), and the output — a ranked "fix this first" queue — maps to a real weekly workflow instead of an abstract prediction. It also gives a clean four-question frame (Section 2) and lets me pull 2-3 real numbers straight from data already in this repo, without needing warehouse access yet.


In [1]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)


(30000, 44)


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** which content items the SEO content lead should prioritize in this week's refresh queue.

**Actor + action:** the SEO content lead reviews the ranked queue and assigns the top items to writers/editors for that week's refresh work.

**Cost of a wrong call:** if the queue over-flags low-impact pages, editors spend hours refreshing content nobody sees, while genuinely high-traffic declining pages sit untouched — a real, measurable loss in search visibility. If the queue under-flags real problems, the team never gets a chance to intervene before further decline.

**Unit of analysis:** one content page (one row in the starter dataset).


In [2]:
# No new computation needed here -- Section 2 is the framing itself.
# The numbers that justify this framing are computed in Section 3.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three numbers, computed below. Note on sample sizes: `freshness_tier` is unevenly distributed
(0-30 days: 20,480 rows; 91-180 days: 9,171 rows; 31-90 and 181+ days: ~175 rows each) -- the two
small tiers are too thin to draw firm conclusions from, so the claim below leans on the
well-populated `91-180` tier, not the noisy `181+` tail.

1. Content stale for 91-180 days shows a higher "down" trend rate (61.1%, n=9,171) than fresh
   content at 0-30 days (51.1%, n=20,480) -- a real, well-supported gap, not a sample-size fluke.
2. A real, sizeable pool of pages is both stale (91+ days) and declining -- a genuine queue, not
   a handful of edge cases.
3. Within that pool, impressions vary hugely -- staleness alone can't tell us which pages matter,
   which is the actual argument for a scored/ranked approach instead of a flat rule.


In [3]:
# 1. Decline rate by freshness tier
decline_rate_by_tier = (
    df.groupby("freshness_tier")["trend_direction"]
      .apply(lambda s: (s == "down").mean())
      .sort_index()
)
print("Share of pages trending 'down', by freshness tier:")
print(decline_rate_by_tier.round(3))
print()

# 2. Size of the stale + declining pool
stale = df[df["freshness_tier"].isin(["91-180", "181+"])]
stale_declining = stale[stale["trend_direction"] == "down"]
print(f"Stale pages (91+ days since update): {len(stale)}")
print(f"Stale AND declining pages: {len(stale_declining)}")
print()

# 3. Impressions spread within the stale+declining pool -- staleness alone isn't enough signal
median_impressions = df["impressions_90d"].median()
high_impact = stale_declining[stale_declining["impressions_90d"] > median_impressions]
pct_high_impact = 100 * len(high_impact) / len(stale_declining)
print(f"Dataset median impressions_90d: {median_impressions}")
print(f"Stale+declining pages ABOVE median impressions: {len(high_impact)} of {len(stale_declining)} ({pct_high_impact:.1f}%)")
print(f"-> roughly {100 - pct_high_impact:.0f}% of a naive stale+declining queue would be low-traffic pages -- ")
print("   confirming that a flat staleness rule alone would waste editor time on low-impact items.")


Share of pages trending 'down', by freshness tier:
freshness_tier
0-30      0.511
181+      0.471
31-90     0.589
91-180    0.611
Name: trend_direction, dtype: float64

Stale pages (91+ days since update): 9345
Stale AND declining pages: 5686

Dataset median impressions_90d: 731.0
Stale+declining pages ABOVE median impressions: 3684 of 5686 (64.8%)
-> roughly 35% of a naive stale+declining queue would be low-traffic pages -- 
   confirming that a flat staleness rule alone would waste editor time on low-impact items.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this work CAN claim:**
- Observed: stale content has a higher observed rate of declining trend than fresh content, in this starter sample.
- Directional: combining staleness with traffic volume (impressions) is a more useful prioritization signal than staleness alone.
- Decision-support: a ranked queue built from these signals can help an editor decide *where to look first* -- not decide *what will happen* if they refresh.

**What this work will NEVER claim:**
- That refreshing a page *causes* recovery -- that requires a real experiment (e.g. before/after with a control group), which this project does not run.
- That any result here reflects or predicts Google's ranking algorithm.
- That `trend_direction` / `trend_pct` are usable as model *features* going forward -- they are the outcome we're trying to explain, not inputs (using them as features would be leakage).


In [4]:
# No additional computation needed -- Section 4 is a claims/scope statement,
# grounded in the numbers already computed in Section 3.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.